# Universal Turing  machine

We will sketch how to construct a universal Turing machine, $U$.

Programming a reasonably large Turing machine involves a number of tasks that are tedious, with fiddly details.  We will try to strike a balance: to say enough that it will be perfectly clear that these tasks can be done, while also avoiding laying out the full collection of four-tuples, which is counterproductive. 

We have boiled the job of writing $U$ down to five tasks
(this approach is derived from a document by Jason Teutsch.).
Where writing a Turing machine is writing machine code, 
these five tasks are like macros for the assembler.
They are a somewhat higher-level description of what their consituent instruction sequences are doing.

We will first describe the input to $U$.
Then we will sketch how $U$ works, except that we will only sketch the five tasks, and leave producing the
four-tuples as exercises.

## The input to $U$

We are given the tape containing the number of the machine $\mathcal{M}$.
Immediately to the right is a marker, $Z$ (see below), and after that is the machine's input.
![This shows the layout](img/utm0.png)
We take the encoding, both of the Turing machine and of the tape characters, to be as in the prior lab.
In particular, we represent integers or characters in unary, as strings of $1$'s.

### Markers

The lab on encoding Turing machines uses the alphabet $\{B,1\}$.
But here we will use *markers*, such as $Z$, or $X$, or $Y$ (our convention is to use capital letters, and to stay away from $B$).

As we saw in the encoding lab, we often need to separate information on the tape.
In that lab, we used single blanks to
separate the parts of a four-tuple instruction, and also used pairs of blanks to separate four-tuples.
Here we need some more separators.

We can imagine that $Z$ stands for a 5-tuple of blanks, and that other markers stand for other sequences.
(We must arrange that there is no possibility of confusing, say, a $Z$ followed by another $Z$ for a ten-blank marker.)
But another option is to instead think of it as having enlarged the alphabet of $U$ to allow new characters. 
This is very convenient, both for readability here and also because it means, for instance,
that we can just put $Z$ in an instruction and don't have to write lots of
four-tuples to work through strings of five blanks. 
In short, we will take the universal machine to have the alphabet 
$\Sigma=\{B,1,G,H,\ldots Z\}$.

## Sketch of $U$'s action

The action of $U$ in computing the effect of the machine $M$
is very much like the Racket program from an earlier lab that simulates a given Turing machine.
At its heart, $U$ iterates $M$'s Delta function.

### Set up a buffer

Before $U$ begins the simulation, it does a bit of bookkeeping.
Remember that the representation of $M$ is really just a list of four-tuples.
We shall use this list to compute the values of $M$'s Delta function.

For that computation, to the left of marker $Y$, we set aside a region of tape as a *buffer* 
to hold the inputs to Delta, that is, to hold the present state and present symbol.
![This shows the buffer](img/utm1.png)
This buffer must be wide enough to hold the number of the largest state and the
largest number representing a symbol, along with a separator between them.
We mark the left of the buffer with $X$.

So, one of the five tasks at the end of this document explains how to search through the representation
of $M$ and find the largest state number or largest character representation.

### Represent the tape

The universal machine $U$ must simulate the tape of $M$ that is infinite in both directions.
We will divide $M$'s tape cells into two groups: the cells to the left of the initial position of $M$'s
read/write head, and the cells to the right, along with the one under the head.
The machine $U$ simulates the cells to the left using the tape region to the left of the marker $X$.
And $U$ simulates the the cell under the intial position along with those to the right 
by using the tape region to the right of the marker $Z$.
![move machine left or right](img/utm3.png)



### Iterate the Delta function

Now we are ready for the heart of the simulation.
Here are the steps.

0. The present state is $0$, and the present character is the one represented to the right of $Z$.
   Put these two unary numbers, 
   the state number and the representation number, into the buffer and separate them with a blank.
   This requires copying a contiguous sequence of $1$'s from one place on the tape 
   to another; this is Task 1 below.

1. Find $M$'s four-tuple with the state, character pair matching what is in the buffer.
   Searching inside the machine for a match is Task 2 below.
   
2. If the next action is `L` or `R` then move the simulated head, as in Task 5.
   Otherwise, copy the next state and next character into the buffer, 
   and return to step 1.
   Doing an if-then is Task 3.


## Five tasks

To run the simulation we will need five tasks, which we can think of
as macros for an assembler, or as routines in a simple programming language.
As sketched above, these are things that $U$ will use in the course of simulating $M$.

These are arranged from easiest to hardest, rather than in the order they are encountered earlier.
This is because experience with easier tasks is a big help with doing later ones.

Each task is described in italics.
After this statement of requirements is a sketch of one possible approach.
(You can ignore the state numbers in the sketch pictures; they are only the ones that happen to appear in one version of the machine.)

### Task1: copying

The universal machine needs to copy numbers from place to place.
For instance, it needs to copy the next state into the buffer.

We show how to copy using markers $S$ and $T$.
This exercise asks for a machine that copies from the left to the right, but a similar machine copies
from right to left (with the added assumption that the space between the two markers is large
enough to hold the source number).

#### Exercise

*Begin with the tape containing markers $S$ and $T$, with $T$ to the right of $S$.
In the interval between the two are strokes and blanks, but the tape is blank to the right of $T$.
Start with the read/write head of $U$ pointing to $S$.*
![copying, setup at the start](img/copy0.png)
*Write a Turing machine, a set of four-tuples, that finishes with the interval unchanged, but also with the number immediately after $S$ (that is, the number of contiguous $1$'s to the right of $S$ with no intervening blanks) copied just to the right of $T$.
End with the head pointing to $S$.*

Here is a sketch of one way to proceed.
From the starting setup, move right and replace the $1$ with a $G$, then move past $T$ and put a $1$ there.
![copying, first pass](img/copy1.png)
Now iterate.
Return to $S$, slide past the existing $G$'s, replace the $1$ that immediately follows with another $G$, and then
move past $T$ along with any subsequent $1$'s, and put down a new $1$.
![copying, subsequent passes](img/copy2.png)
When the iteration in the prior sentence goes to the end of the $G$'s and finds a $B$, return to the
starting $S$ and change the $G$'s to $1$'s.
![copying, finished](img/copy3.png)

### Task 2: matching

The universal machine $U$ needs to match the current state and current symbol with the four-tuple instructions 
stored in the description of $M$.
Then it can fetch the next state and the next action.

#### Exercise

*Let $S$ and $T$ be markers, with $S$ to the left of $T$, and the other characters on the tape are only blanks and $1$'s.
Start with the read/write head of the machine on the marker $S$.
![matching, setup at the start](img/match0.png)
Write a Turing machine that ends in state $q_{100}$ if the number of $1$'s immediately following $S$ equals the number
of $1$'s immediately following $T$, that is, if $x_s=x_t$.
If the number of $1$'s don't match then the machine ends in state $q_{101}$.
The machine should end with the tape exactly the same as when it started, and with the head pointing to
marker $S$.*

One way to proceed is to start at $S$ and for each $1$, rewrite it as a $G$.
Then scan right past $T$ to find the matching $1$ and rewrite it as an $H$.
![matching, pair the 1s](img/match1.png)
Iterate this until one or the other sequence of $1$'s runs out.
Here we show the case where the $1$'s after $S$ are the first to run out.
![matching, 1s after S run out](img/match2.png)
In this case we scan right to find whether the $1$'s after $T$ have run out at the same time.
![matching, 1s after T ran out at same time](img/match3.png)
In this example they have run out at the same time, so $x_s=x_t$. 
Change the $G$'s and $H$'s back to $1$'s, reposition the head at $S$, and go to state $q_{100}$.
![matching, finish in state q100](img/match4.png)

### Task 3: substitution

Let $S$ and $T$ be markers, with $S$ to the left of $T$, and only blanks and $1$'s between.
Start with the head of $U$ on the marker $S$.

Let $I$ be a sequence of unary representations, that is, sequences of $1$'s, separated by single blanks.
Suppose that $I$ lies immediately to the right of $T$.
Suppose also that there is a single unary representation, with $k$-many $1$'s, to the immediate right of $S$.

![the before picture](img/utm5.png)

Then there is a set of Turing four-tuples that replaces the first term in $I$, its first unary representation,
with $k$-many $1$'s.

![the after picture](img/utm6.png)

Note that in replacing three $1$'s with two, the machine must close up, so that $I$ is one shorter.

#### Exercise

### Task 4: establish the buffer

We need to leave enough space to hold the largest number that we find.
(Actually, we must leave room for the largest state and also room for the largest character representation.
But here we will just do the single largest number in the interval.)

#### Exercise

*Begin with the tape having markers $S$ and $T$, with $T$ to the right of $S$.
In the interval between the two are strokes and blanks, but the tape is blank to the left of $S$ as well as to the right of $T$.
Start with the head pointing at $S$.*
![setup at the start](img/findmax0.png)
*Write a Turing machine that finishes with the interval unchanged, and with a copy of the largest number from
that interval just to $S$'s left.*


One way to proceed is to move right from $S$, find the first number, and replace $1$'s with $H$'s. 
![replace 1's with H's](img/findmax1.png)
For each such replacement, move left from $S$ and put $G$'s over what is there.
![remember each H with a G to left of S](img/findmax2.png)
At the end of the first number to $S$'s right, change the blank to a $J$. 
![replace B's with J's](img/findmax3.png)
Then go back and turn the $G$'s to $1$'s.
![replace G's with 1's](img/findmax4.png)

Now iterate the prior steps.
The reason to use $G$'s instead of directly using $1$'s becomes clear; we can see that the second number
to $S$'s right is smaller than the first because the $G$'s sit inside the $1$'s from the prior iteration.
![iterate prior steps](img/findmax5.png)

This is the picture when we are done.
![Done](img/findmax6.png)


### Task 5: move the simulated head

For $U$ to simulate $M$, it must simulate tape moves. 
Here we show how $U$ can slide past the character represented under $M$'s head, that is, the first
sequence of $1$'s to the right of $Z$.

#### Exercise 

*Begin with a tape containing the markers $S$ and $T$, with $S$ to the left of $T$.
All other symbols on the tape are blanks and $1$'s.
To the right of $T$ is a blank, then a sequence of $1$'s, and then
another blank (possibly the sequence of $1$'s is empty).
Call the sequence of $1$'s the "representation under $U$'s head".
Here it is marked $x$.
![move machine right, setup](img/move0.png)
Write a Turing machine that moves the representation under $U$'s head to the left of $S$.
That is, when the machine finishes the the initial blank and the sequence of $1$'s is no longer to the
right of $T$.
Instead, to the left of $S$ the machine has inserted a sequence of the same number of $1$'s, separated
from $S$ by a blank.
In terms of the above picture, when the code is finished then `11` should be to the left of $S$, and no longer to the right of $T$.*

You could proceed by first sliding right to $T$,
and overwriting the blank that is there.
![move machine right, grab blank separator](img/move1.png)
Then we move the other things in the interval to the right,
or what is the same thing, carrying the blank down until it is just to the left of $S$
![move machine right, move blank separator down to left of S](img/move2.png)

The next step is the critical one.
Slide down to $T$ again.
The are two cases.
If on $T$'s right is a blank then the character represented on $M$'s tape is a blank.
Finish by carrying it down to the left of $S$.

The other case is that the character to the right of $T$ is a $1$.
In this case we carry each such $1$ down to the left of $S$,
![move machine right, move a 1 down to left of S](img/move3.png)
and repeating until all the $1$'s are moved, that is, until the next character to the right of $T$ is a blank.
![move machine right, done](img/move4.png)